# Final Testcase Generator — Kaggle smoke test

Bật GPU và Internet trước khi chạy. Backend `native` không phải security sandbox; chỉ dùng với đề tin cậy để kiểm tra luồng.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, time, urllib.request

SOURCE = Path('/kaggle/input/final-testcase-generator')  # sửa theo dataset đã attach
PROJECT = Path('/kaggle/working/Final_TestcaseGenerator')
PROBLEM = 'p02548'  # sửa thành bài đã có statement.md
MODEL = 'gemma3:4b'

if not (SOURCE / 'main.py').is_file():
    raise FileNotFoundError(f'Sửa SOURCE; không tìm thấy main.py tại {SOURCE}')
shutil.copytree(
    SOURCE, PROJECT, dirs_exist_ok=True,
    ignore=shutil.ignore_patterns('.venv', 'output', '__pycache__', '*.pyc'),
)
os.chdir(PROJECT)
print(PROJECT)

In [ ]:
# Ollama không cần Python package; pipeline gọi HTTP API bằng standard library.
if shutil.which('ollama') is None:
    subprocess.run(
        ['bash', '-lc', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True,
    )
os.environ['OLLAMA_CONTEXT_LENGTH'] = '32768'
ollama_log = open('/kaggle/working/ollama.log', 'ab', buffering=0)
ollama = subprocess.Popen(
    ['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT,
    start_new_session=True,
)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).close()
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError('Ollama không khởi động; xem /kaggle/working/ollama.log')
subprocess.run(['ollama', 'pull', MODEL], check=True)

In [ ]:
subprocess.run(
    [
        'python', 'scripts/kaggle_preflight.py', '--problem', PROBLEM,
        '--model', MODEL, '--runtime-config', 'config/kaggle.runtimes.json',
    ],
    check=True,
)

In [ ]:
RUN_OUTPUT = Path('/kaggle/working/testgen-output') / PROBLEM
command = [
    'python', 'main.py', '--problem', PROBLEM,
    '--config', 'config/kaggle.pipeline.json',
    '--judge-config', 'config/kaggle.judge.json',
    '--runtime-config', 'config/kaggle.runtimes.json',
    '--judge', 'native', '--allow-unsafe-native-judge',
    '--ollama-model', MODEL, '--ollama-fallback-model', MODEL,
    '--output', str(RUN_OUTPUT),
]
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
archive = shutil.make_archive(
    '/kaggle/working/testgen-output', 'zip', '/kaggle/working/testgen-output'
)
print('Tải artifact tại:', archive)